## Idea

For this notebook, we will follow the v4 temporal strategy:

train_v4.parquet contains days 1–12
test_v4.parquet contains days 13–15 (untouched)
Training uses expanding rolling validation
Each fold: trains on past days only, validates on the next day, logs parameters/metrics to MLflow
No test evaluation is performed here.

---

### Imports

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

import xgboost as xgb

import mlflow
import mlflow.xgboost

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

### Paths and MLFlow

In [4]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_PATH / "dvc" / "train_v4.parquet"
BEST_PARAMS_PATH = DATA_PATH / "results" / "best_xgb_params.json"

mlflow.set_tracking_uri(
    f"file:{PROJECT_ROOT / 'mlruns'}"
)

mlflow.set_experiment(
    "simbox_fraud_detection_v4"
)

c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/07/26 00:48:54 INFO mlflow.tracking.fluent: Experiment with name 'simbox_fraud_detection_v4' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:\\Users\\ali.farhat\\Documents\\simbox-project\\mlruns/166962124169963518', creation_time=1785016134603, experiment_id='166962124169963518', last_update_time=1785016134603, lifecycle_stage='active', name='simbox_fraud_detection_v4', tags={}, trace_location=None, workspace='default'>

### Load dataset and hyperparameters

In [5]:
train_df = pd.read_parquet(TRAIN_PATH)

train_df["profile_date"] = pd.to_datetime(
    train_df["profile_date"]
)

with open(BEST_PARAMS_PATH, "r") as f:
    params = json.load(f)

params.update({
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "seed": 42,
    "n_jobs": -1
})

# scale_pos_weight is recomputed per fold
params.pop("scale_pos_weight", None)

print(train_df.shape)

train_df.head()

(343365, 15)


,msisdn,profile_date,repeat_call_ratio,flagged_outgoing_ratio,incoming_outgoing_ratio,incoming_calls,avg_outgoing_duration,flagged_calls,unique_callees,short_call_ratio,avg_incoming_calls_per_hour,avg_outgoing_calls_per_hour,cell_usage_entropy,period_entropy,label
0,22362444330,2022-11-01,0.800000,999999.000000,999999.000000,5,0.000000,5,0,0.000000,1.666667,0.000000,0.721928,1.521928,0
1,2250152130667,2022-11-01,0.000000,1.000000,0.000000,0,39.000000,3,3,0.666667,0.000000,3.000000,0.918296,-0.000000,0
2,2250152127834,2022-11-01,0.250000,1.333333,0.333333,3,81.777778,12,8,0.416667,1.500000,1.500000,2.522055,1.280672,1
3,2250152127796,2022-11-01,0.742857,1.428571,1.500000,21,69.857143,20,8,0.285714,1.615385,2.333333,2.069383,2.167014,0
4,2250152127414,2022-11-01,0.000000,999999.000000,999999.000000,1,0.000000,1,0,0.000000,1.000000,0.000000,-0.000000,-0.000000,0


### Prepare dataset

In [6]:
train_df = train_df.set_index(
    ["msisdn", "profile_date"]
)

X = train_df.drop(
    columns=["label"]
)

y = train_df["label"]

print(X.shape)

(343365, 12)


---
### Define rolling validation folds

In [8]:
unique_days = sorted(
    X.index.get_level_values("profile_date").unique()
)

validation_days = unique_days[-4:]

print("Rolling validation:")

for day in validation_days:
    train_days = [
        d for d in unique_days
        if d < day
    ]

    print(
        f"Train: {train_days[0].date()} → {train_days[-1].date()} | "
        f"Validation: {day.date()}"
    )

Rolling validation:
Train: 2022-11-01 → 2022-11-08 | Validation: 2022-11-09
Train: 2022-11-01 → 2022-11-09 | Validation: 2022-11-10
Train: 2022-11-01 → 2022-11-10 | Validation: 2022-11-11
Train: 2022-11-01 → 2022-11-11 | Validation: 2022-11-12


### Rolling training and validation

In [9]:
fold_results = []
models = []

with mlflow.start_run(
    run_name="rolling_validation_xgb_v4"
):

    mlflow.log_params(params)

    for fold, val_day in enumerate(validation_days, start=1):

        train_days = [
            d for d in unique_days
            if d < val_day
        ]

        train_mask = (
            X.index.get_level_values("profile_date")
            .isin(train_days)
        )

        val_mask = (
            X.index.get_level_values("profile_date")
            == val_day
        )

        X_train = X.loc[train_mask]
        y_train = y.loc[train_mask]

        X_val = X.loc[val_mask]
        y_val = y.loc[val_mask]

        scale_pos_weight = (
            (y_train == 0).sum()
            /
            max((y_train == 1).sum(), 1)
        )

        fold_params = params.copy()
        fold_params["scale_pos_weight"] = scale_pos_weight

        dtrain = xgb.DMatrix(
            X_train,
            label=y_train
        )

        dval = xgb.DMatrix(
            X_val,
            label=y_val
        )

        model = xgb.train(
            params=fold_params,
            dtrain=dtrain,
            num_boost_round=500,
            evals=[(dval, "validation")],
            verbose_eval=False
        )

        val_pred = model.predict(dval)

        pr_auc = average_precision_score(
            y_val,
            val_pred
        )

        roc_auc = roc_auc_score(
            y_val,
            val_pred
        )

        print(
            f"\nFold {fold}"
        )

        print(
            f"Train: {train_days[0].date()} → {train_days[-1].date()}"
        )

        print(
            f"Validation: {val_day.date()}"
        )

        print(
            f"PR-AUC: {pr_auc:.4f}"
        )

        print(
            f"ROC-AUC: {roc_auc:.4f}"
        )

        fold_results.append({
            "fold": fold,
            "train_start": train_days[0],
            "train_end": train_days[-1],
            "validation_day": val_day,
            "train_samples": len(X_train),
            "validation_samples": len(X_val),
            "scale_pos_weight": scale_pos_weight,
            "pr_auc": pr_auc,
            "roc_auc": roc_auc
        })

        models.append(model)

        mlflow.log_metric(
            f"fold_{fold}_pr_auc",
            pr_auc
        )

        mlflow.log_metric(
            f"fold_{fold}_roc_auc",
            roc_auc
        )

    mlflow.log_metric(
        "mean_pr_auc",
        np.mean(
            [x["pr_auc"] for x in fold_results]
        )
    )

    mlflow.log_metric(
        "mean_roc_auc",
        np.mean(
            [x["roc_auc"] for x in fold_results]
        )
    )

2026/07/26 00:49:28 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\xgboost\training.py:200: User


Fold 1
Train: 2022-11-01 → 2022-11-08
Validation: 2022-11-09
PR-AUC: 0.3936
ROC-AUC: 0.8421


c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:49:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "num_boost_round", "params" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Fold 2
Train: 2022-11-01 → 2022-11-09
Validation: 2022-11-10
PR-AUC: 0.3937
ROC-AUC: 0.8405


c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:49:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "num_boost_round", "params" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Fold 3
Train: 2022-11-01 → 2022-11-10
Validation: 2022-11-11
PR-AUC: 0.4317
ROC-AUC: 0.8561


c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:49:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "num_boost_round", "params" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Fold 4
Train: 2022-11-01 → 2022-11-11
Validation: 2022-11-12
PR-AUC: 0.3918
ROC-AUC: 0.8535


### Validation results table

In [10]:
results_df = pd.DataFrame(
    fold_results
)

display(results_df)

,fold,train_start,train_end,validation_day,train_samples,validation_samples,scale_pos_weight,pr_auc,roc_auc
0,1,2022-11-01,2022-11-08,2022-11-09,228428,28761,16.493337,0.393591,0.842057
1,2,2022-11-01,2022-11-09,2022-11-10,257189,28535,16.437725,0.393679,0.840532
2,3,2022-11-01,2022-11-10,2022-11-11,285724,28886,16.400974,0.431707,0.856123
3,4,2022-11-01,2022-11-11,2022-11-12,314610,28755,16.353963,0.391828,0.853472


### Overall validation score

In [11]:
print(
    "Mean PR-AUC:",
    results_df["pr_auc"].mean()
)

print(
    "Std PR-AUC:",
    results_df["pr_auc"].std()
)

print(
    "Mean ROC-AUC:",
    results_df["roc_auc"].mean()
)

print(
    "Std ROC-AUC:",
    results_df["roc_auc"].std()
)

Mean PR-AUC: 0.4027014015426116
Std PR-AUC: 0.019355637118077702
Mean ROC-AUC: 0.8480461492859592
Std ROC-AUC: 0.007895492900261938


---
### Save final model candidates

<i> (may be done later)</i>